# Weight Window Inspector

In [1]:
from kika.wwinp import read_wwinp
import numpy as np

FILEPATH = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/wwinp/wwinp_1"
ww = read_wwinp(FILEPATH)

fm = ww.geometry.fine_mesh
e_bins = ww.energy_bins[0]
arr = ww.values.ww_values[0]  # (nt, ne, nfz, nfy, nfx)
h = ww.header

print(ww)

WWINP  mesh_type=cartesian  particles=1  time_dep=no
  file: /mnt/c/Users/MONLEON-DE-LA-JAN/Documents/wwinp/wwinp_1
  Geometry:
    x: [-442, 440]  bins=182
    y: [-440, 445]  bins=216
    z: [-400, 400]  bins=120
  Particle 0:
    shape=(1, 5, 120, 216, 182)  energies=5
    min=7.516e-11  max=8.395e+19  nonzero=23587200/23587200 (100.0%)


## Domain limits

In [2]:
for axis in ("x", "y", "z"):
    g = fm[axis]
    print(f"{axis}: [{g[0]:.2f}, {g[-1]:.2f}]  ({len(g)-1} bins)")

print(f"\nEnergy (MeV): {len(e_bins)} bins")
for i, e in enumerate(e_bins):
    lo = 0.0 if i == 0 else e_bins[i - 1]
    print(f"  bin {i}: [{lo:.6e}, {e:.6e}]")

x: [-442.00, 440.00]  (182 bins)
y: [-440.00, 445.00]  (216 bins)
z: [-400.00, 400.00]  (120 bins)

Energy (MeV): 5 bins
  bin 0: [0.000000e+00, 1.422700e+00]
  bin 1: [1.422700e+00, 1.826800e+00]
  bin 2: [1.826800e+00, 3.011900e+00]
  bin 3: [3.011900e+00, 6.376300e+00]
  bin 4: [6.376300e+00, 2.000000e+01]


## Query a point

Set `x`, `y`, `z` (cm) and `energy` (MeV) to get the weight window value.

In [22]:
# ── Set your query here ───────────────────────────────────────
x = 150.0      # cm
y = 0.0        # cm
z = 0.0        # cm
energy = 2   # MeV
# ──────────────────────────────────────────────────────────────

def lookup_ww(x, y, z, energy):
    """Return the weight window value at (x, y, z, energy)."""
    # Find spatial bin indices
    ix = int(np.clip(np.searchsorted(fm["x"], x, side="right") - 1, 0, h.nfx - 1))
    iy = int(np.clip(np.searchsorted(fm["y"], y, side="right") - 1, 0, h.nfy - 1))
    iz = int(np.clip(np.searchsorted(fm["z"], z, side="right") - 1, 0, h.nfz - 1))

    # Find energy bin index
    e_upper = np.array(e_bins)
    ie = int(np.searchsorted(e_upper, energy, side="left"))
    ie = int(np.clip(ie, 0, len(e_bins) - 1))

    e_lo = 0.0 if ie == 0 else e_bins[ie - 1]
    e_hi = e_bins[ie]
    val = arr[0, ie, iz, iy, ix]

    print(f"Point: ({x}, {y}, {z}) cm,  E = {energy} MeV")
    print(f"  x bin: [{fm['x'][ix]:.2f}, {fm['x'][ix+1]:.2f}]")
    print(f"  y bin: [{fm['y'][iy]:.2f}, {fm['y'][iy+1]:.2f}]")
    print(f"  z bin: [{fm['z'][iz]:.2f}, {fm['z'][iz+1]:.2f}]")
    print(f"  E bin: [{e_lo:.6e}, {e_hi:.6e}] MeV")
    print(f"  WW value = {val:.6e}")
    return val

_ = lookup_ww(x, y, z, energy)

Point: (150.0, 0.0, 0.0) cm,  E = 2 MeV
  x bin: [150.00, 152.00]
  y bin: [-1.00, 1.00]
  z bin: [0.00, 5.00]
  E bin: [1.826800e+00, 3.011900e+00] MeV
  WW value = 2.643850e-03


In [20]:
1/2.643850e-03

378.23628420674396